# CS5720 Home Assignment 4

**Jeff Agnitsch** · Student ID: 7005146960 · Fall 2026

Part II code (NLP preprocessing, scaled dot-product attention) and a numeric check of the Part III hand calculation. Part I and Part III write-ups, and the Part II short answers, are in `README.md`. The same code lives as standalone scripts in `src/`.

## Part II, Q1: NLP preprocessing pipeline

Tokenize the sentence, remove English stop words, then stem each remaining token. Uses NLTK, the library from the Chapter 7 lecture. The first run downloads the small `punkt_tab` and `stopwords` resources.

In [1]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# The sentence the assignment asks us to process.
SENTENCE = "NLP techniques are used in virtual assistants like Alexa and Siri."


def ensure_nltk_data() -> None:
    """Download the NLTK resources this script needs, if they are not present.

    `punkt_tab` holds the sentence/word tokenizer model and `stopwords` holds the
    English stop word list. The downloads are small and only happen once; after that
    nltk.data.find() succeeds and nothing is fetched.
    """
    for resource, path in [("punkt_tab", "tokenizers/punkt_tab"),
                           ("stopwords", "corpora/stopwords")]:
        try:
            nltk.data.find(path)
        except LookupError:
            nltk.download(resource, quiet=True)


def preprocess(sentence: str) -> tuple[list[str], list[str], list[str]]:
    """Tokenize, remove stop words, and stem a sentence.

    Returns the three intermediate lists so the caller can print or test each stage.
    The stop word check lowercases each token before looking it up, because the NLTK
    list is all lowercase and we do not want a capitalized "The" to slip through.
    Punctuation is left in on purpose: the assignment's expected output describes
    the first list as "all words and punctuation", and the lecture example kept the
    trailing "!" as well.
    """
    # Step 1 - tokenize. word_tokenize splits on whitespace and also peels
    # punctuation off into its own token, so the final "." becomes a separate item.
    tokens = word_tokenize(sentence)

    # Step 2 - remove stop words. These are high-frequency function words
    # ("are", "in", "and", "like") that carry little topical meaning on their own.
    stop_words = set(stopwords.words("english"))
    tokens_no_stop = [tok for tok in tokens if tok.lower() not in stop_words]

    # Step 3 - stem. The Porter stemmer strips suffixes with fixed rules, so
    # "techniques" -> "techniqu" and "assistants" -> "assist". The result is not
    # always a real dictionary word; that is the trade-off versus lemmatization.
    stemmer = PorterStemmer()
    stemmed = [stemmer.stem(tok) for tok in tokens_no_stop]

    return tokens, tokens_no_stop, stemmed


def main() -> None:
    ensure_nltk_data()
    tokens, tokens_no_stop, stemmed = preprocess(SENTENCE)

    print("Sentence:", SENTENCE)
    print()
    print("1. Original Tokens:          ", tokens)
    print("2. Tokens Without Stopwords: ", tokens_no_stop)
    print("3. Stemmed Words:            ", stemmed)

    # Show which words the stop word filter actually removed, so the effect of
    # step 2 is visible rather than implied.
    removed = [tok for tok in tokens if tok not in tokens_no_stop]
    print()
    print("Stop words removed:", removed)


main()


Sentence: NLP techniques are used in virtual assistants like Alexa and Siri.

1. Original Tokens:           ['NLP', 'techniques', 'are', 'used', 'in', 'virtual', 'assistants', 'like', 'Alexa', 'and', 'Siri', '.']
2. Tokens Without Stopwords:  ['NLP', 'techniques', 'used', 'virtual', 'assistants', 'like', 'Alexa', 'Siri', '.']
3. Stemmed Words:             ['nlp', 'techniqu', 'use', 'virtual', 'assist', 'like', 'alexa', 'siri', '.']

Stop words removed: ['are', 'in', 'and']


## Part II, Q2: Scaled dot-product attention

$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$

Each intermediate matrix is printed so the four steps can be checked by hand.

In [2]:
import numpy as np

# Print floats with four decimals and no scientific notation, so the matrices
# are easy to read and compare against hand calculations.
np.set_printoptions(precision=4, suppress=True)


def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Numerically stable softmax along `axis`.

    Subtracting the row maximum before exponentiating does not change the result
    (softmax is shift-invariant) but keeps exp() from overflowing on large scores.
    """
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q: np.ndarray, K: np.ndarray, V: np.ndarray,
                                 verbose: bool = True) -> tuple[np.ndarray, np.ndarray]:
    """Compute scaled dot-product attention.

    Q: (n_queries, d_k)   one query vector per row
    K: (n_keys,    d_k)   one key vector per row, same width as Q
    V: (n_keys,    d_v)   one value vector per row, one per key

    Returns (attention_weights, output) where
        attention_weights has shape (n_queries, n_keys) and each row sums to 1
        output            has shape (n_queries, d_v)
    """
    Q, K, V = (np.asarray(m, dtype=float) for m in (Q, K, V))
    d_k = K.shape[1]  # the key dimension used for scaling

    # Step 1 - raw similarity between every query and every key.
    scores = Q @ K.T

    # Step 2 - scale by sqrt(d_k). Dot products grow with the vector length, and
    # large scores push softmax toward a one-hot output whose gradient is near zero.
    # Dividing by sqrt(d_k) keeps the scores at a size softmax handles well.
    scaled_scores = scores / np.sqrt(d_k)

    # Step 3 - softmax over the keys (axis=-1) turns each row into a probability
    # distribution: how much this query should attend to each key.
    weights = softmax(scaled_scores, axis=-1)

    # Step 4 - each output row is a weighted average of the value rows.
    output = weights @ V

    if verbose:
        print(f"Q K^T (raw scores):\n{scores}\n")
        print(f"Scaled by sqrt(d_k) = sqrt({d_k}) = {np.sqrt(d_k):.4f}:\n{scaled_scores}\n")
        print(f"Attention weights (softmax over each row):\n{weights}")
        print(f"Row sums: {weights.sum(axis=1)}\n")
        print(f"Output = weights @ V:\n{output}\n")

    return weights, output


def main() -> None:
    # Test inputs given in the assignment. Q and K are identical, and each query is
    # orthogonal to the other key, so the score matrix is diagonal.
    Q = np.array([[1, 0, 1, 0],
                  [0, 1, 0, 1]])
    K = np.array([[1, 0, 1, 0],
                  [0, 1, 0, 1]])
    V = np.array([[1, 2, 3, 4],
                  [5, 6, 7, 8]])

    print("=== Assignment test case ===\n")
    weights, output = scaled_dot_product_attention(Q, K, V)

    print("Final answer")
    print("Attention weights:\n", weights)
    print("Output:\n", output)


main()


=== Assignment test case ===

Q K^T (raw scores):
[[2. 0.]
 [0. 2.]]

Scaled by sqrt(d_k) = sqrt(4) = 2.0000:
[[1. 0.]
 [0. 1.]]

Attention weights (softmax over each row):
[[0.7311 0.2689]
 [0.2689 0.7311]]
Row sums: [1. 1.]

Output = weights @ V:
[[2.0758 3.0758 4.0758 5.0758]
 [3.9242 4.9242 5.9242 6.9242]]

Final answer
Attention weights:
 [[0.7311 0.2689]
 [0.2689 0.7311]]
Output:
 [[2.0758 3.0758 4.0758 5.0758]
 [3.9242 4.9242 5.9242 6.9242]]


## Part III check: lecture self-attention example

Reproduces the three-token example from the Chapter 9 slides with the lecture's recipe: unscaled scores, softmax, round weights to one decimal, then the weighted sum of values. The step-by-step hand calculation for token 3 is in the README; this cell confirms the arithmetic and then runs the same $Q, K, V$ through the scaled function above for comparison.

In [3]:
import numpy as np


np.set_printoptions(precision=4, suppress=True)

# --- Inputs from the lecture slides -------------------------------------------
# Three input tokens, each a 4-dimensional embedding (one row per token).
X = np.array([[1, 0, 1, 0],   # input 1
              [0, 2, 0, 2],   # input 2
              [1, 1, 1, 1]])  # input 3

# Projection matrices (4 x 3): rows index the input dimensions, columns the
# query/key/value dimensions. These are the matrices shown on the slides.
W_Q = np.array([[1, 0, 1],
                [1, 0, 0],
                [0, 0, 1],
                [0, 1, 1]])
W_K = np.array([[0, 0, 1],
                [1, 1, 0],
                [0, 1, 0],
                [1, 1, 0]])
W_V = np.array([[0, 2, 0],
                [0, 3, 0],
                [1, 0, 3],
                [1, 1, 0]])


def lecture_self_attention(X, W_Q, W_K, W_V, token_index: int) -> np.ndarray:
    """Self-attention for one token, following the lecture's unscaled, rounded recipe."""
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    q = Q[token_index]

    # Unscaled scores: dot product of this token's query with every key.
    scores = K @ q
    print(f"Query q{token_index + 1} = {q}")
    for j, (k, s) in enumerate(zip(K, scores), start=1):
        terms = " + ".join(f"{a}*{b}" for a, b in zip(q, k))
        print(f"  score vs key {j}: {q} . {k} = {terms} = {s}")

    # Softmax, then round to one decimal place as done in class.
    weights_exact = softmax(scores.astype(float))
    weights = np.round(weights_exact, 1)
    print(f"softmax({scores}) = {weights_exact}  ->  rounded to 1 dp: {weights}")

    # Weighted values and their sum.
    weighted = weights[:, None] * V
    for j, row in enumerate(weighted, start=1):
        print(f"  {weights[j - 1]} * v{j} {V[j - 1]} = {row}")
    out = weighted.sum(axis=0)
    print(f"New representation of token {token_index + 1}: {out}\n")
    return out


def main() -> None:
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    print("Queries (rows):\n", Q)
    print("Keys (rows):\n", K)
    print("Values (rows):\n", V)
    print()

    print("=== Lecture recipe (unscaled, weights rounded to 1 dp) ===\n")
    results = [lecture_self_attention(X, W_Q, W_K, W_V, i) for i in range(3)]
    print("All three outputs:")
    print(np.array(results))
    print("Slide 41 shows: [2.0 7.0 1.5], [2.0 8.0 0.0], [2.0 7.8 0.3]")
    # Token 3 (the one the assignment asks for) matches the slide exactly. Token 1
    # differs slightly because the slide treats softmax([2, 4, 4]) as [0.0, 0.5, 0.5];
    # the true first weight is 0.063, which rounds to 0.1, not 0.0. The slide notes
    # this approximation itself ("can arise from rounding").
    print()

    # For comparison, run the same Q, K, V through the Part II implementation,
    # which scales by sqrt(d_k) and does not round. The ranking of the weights is
    # unchanged; the scaled version is just a little less peaked.
    print("=== Same inputs through the Part II scaled function (no rounding) ===\n")
    scaled_dot_product_attention(Q, K, V)


main()


Queries (rows):
 [[1 0 2]
 [2 2 2]
 [2 1 3]]
Keys (rows):
 [[0 1 1]
 [4 4 0]
 [2 3 1]]
Values (rows):
 [[1 2 3]
 [2 8 0]
 [2 6 3]]

=== Lecture recipe (unscaled, weights rounded to 1 dp) ===

Query q1 = [1 0 2]
  score vs key 1: [1 0 2] . [0 1 1] = 1*0 + 0*1 + 2*1 = 2
  score vs key 2: [1 0 2] . [4 4 0] = 1*4 + 0*4 + 2*0 = 4
  score vs key 3: [1 0 2] . [2 3 1] = 1*2 + 0*3 + 2*1 = 4
softmax([2 4 4]) = [0.0634 0.4683 0.4683]  ->  rounded to 1 dp: [0.1 0.5 0.5]
  0.1 * v1 [1 2 3] = [0.1 0.2 0.3]
  0.5 * v2 [2 8 0] = [1. 4. 0.]
  0.5 * v3 [2 6 3] = [1.  3.  1.5]
New representation of token 1: [2.1 7.2 1.8]

Query q2 = [2 2 2]
  score vs key 1: [2 2 2] . [0 1 1] = 2*0 + 2*1 + 2*1 = 4
  score vs key 2: [2 2 2] . [4 4 0] = 2*4 + 2*4 + 2*0 = 16
  score vs key 3: [2 2 2] . [2 3 1] = 2*2 + 2*3 + 2*1 = 12
softmax([ 4 16 12]) = [0.    0.982 0.018]  ->  rounded to 1 dp: [0. 1. 0.]
  0.0 * v1 [1 2 3] = [0. 0. 0.]
  1.0 * v2 [2 8 0] = [2. 8. 0.]
  0.0 * v3 [2 6 3] = [0. 0. 0.]
New representation of t